# 카카오 4대 약관 RAG 평가기 (1단계)

문항, 정답 기준(gold), 팀별 답변 파일을 입력받아 0~100점을 매기는 평가기입니다.

- 판정 모델: Gemini 3.5 Flash 단일 모델
- 배점: 근거 조항(MRR) 20 + 내용(F1) 30 + LLM 평가 50
- 문항 수·팀 수·qid는 코드에 고정하지 않고 입력 파일 기준으로 동적 처리
- API 키는 이 노트북 어디에도 직접 쓰지 않고 환경변수/Colab Secrets에서만 읽습니다

**셀은 반드시 위에서 아래로 순서대로 실행하세요.**

## 1. 설치

필요한 패키지를 설치합니다. 가상환경 여부를 자동으로 감지해서 `--break-system-packages` 플래그를 필요할 때만 붙입니다.

In [ ]:
import subprocess
import sys


def _pip_install(*packages: str) -> None:
    """
    가상환경(venv/conda) 안에서는 --break-system-packages가 지원되지 않아 오류가 나고,
    Colab처럼 시스템 전역 Python(PEP 668 외부관리 환경)에서는 이 플래그가 필요하다.
    sys.prefix와 sys.base_prefix를 비교해 가상환경 여부를 감지한다.
    """
    in_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix)
    cmd = [sys.executable, "-m", "pip", "install", "-q"]
    if not in_venv:
        cmd.append("--break-system-packages")
    cmd.extend(packages)

    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError:
        # 위 플래그 자체를 모르는 구버전 pip일 수 있으니 플래그 없이 한 번 더 시도
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)


_pip_install("pydantic>=2", "google-genai")
print("설치 완료")

## 2. API 키 주입

`GOOGLE_API_KEY`는 노트북 코드에 직접 쓰지 않습니다.

- Colab: 왼쪽 열쇠 아이콘(Secrets)에 `GOOGLE_API_KEY`를 등록해두면 아래 셀이 자동으로 읽어옵니다.
- 로컬: 노트북을 실행하기 *전에* 터미널에서 `export GOOGLE_API_KEY="..."` 후 같은 셸에서 `jupyter lab`을 켜면 됩니다.
  (또는 이 셀 실행 전에 `%env GOOGLE_API_KEY=...` 매직 명령을 별도 셀에서 직접 실행해도 됩니다. 이 노트북 파일 자체에는 키 값을 저장하지 마세요.)

In [ ]:
import os

try:
    from google.colab import userdata  # type: ignore
    key = userdata.get("GOOGLE_API_KEY")
    if key:
        os.environ.setdefault("GOOGLE_API_KEY", key)
except Exception:
    pass  # Colab이 아니거나 Secrets가 없으면 무시 — 로컬은 환경변수로 주입

if os.environ.get("GOOGLE_API_KEY"):
    print("GOOGLE_API_KEY 확인됨")
else:
    print("[경고] GOOGLE_API_KEY가 없습니다. LLM 채점은 모두 실패하고 근거+내용 50점 기준 partial로 처리됩니다.")

## 3. 설정

입력 파일 경로 패턴과 실행 옵션입니다. 문항 수·팀 수는 여전히 하드코딩하지 않고,
아래 glob 패턴으로 파일을 찾습니다.

In [ ]:
import os

# 골드셋 파일 패턴 (공개 10문항 개발 단계 기준 기본값)
GOLD_GLOB = os.environ.get("GOLD_GLOB", "gold_questions_public10*.json")

# 답변 파일 패턴. 3일차 실전에는 "answer_BLIND*.json" 등으로 바꾸세요.
ANSWER_GLOB = os.environ.get("ANSWER_GLOB", "answers_public_*.json")

OUTPUT_PATH = os.environ.get("EVAL_OUTPUT_PATH", "eval_output.json")

# 개발 단계 공개 답변 파일은 blind_id가 아니라 team 필드("3" 등)만 가지고 있어
# BLIND01~05 형식이 아니다. 로컬 테스트 시에만 이 값으로 강제 오버라이드한다.
# 실전(비공개 30문항)에서는 운영진이 BLIND01~05로 명명된 파일을 배포하므로
# 이 값을 반드시 None으로 비워둔다.
DEV_BLIND_ID_OVERRIDE = os.environ.get("DEV_BLIND_ID_OVERRIDE")  # 예: "BLIND01"

print(f"GOLD_GLOB={GOLD_GLOB!r}")
print(f"ANSWER_GLOB={ANSWER_GLOB!r}")
print(f"OUTPUT_PATH={OUTPUT_PATH!r}")
print(f"DEV_BLIND_ID_OVERRIDE={DEV_BLIND_ID_OVERRIDE!r}")

## 4. 출력 계약 스키마

지침 4장 요구사항을 코드로 강제합니다.
- 최상위 키는 `results`만 (`rank`, `schema_version` 금지)
- `total`은 반올림하지 않은 0~100 float, `failed`일 때만 `None`
- `status`는 `completed` / `partial` / `failed` 세 값만 허용

In [ ]:
from typing import Literal, Optional

from pydantic import BaseModel, ConfigDict, Field


class ResultItem(BaseModel):
    """results 배열의 항목 하나. extra="forbid"로 금지 필드 혼입을 원천 차단."""

    model_config = ConfigDict(extra="forbid")

    blind_id: str = Field(pattern=r"^BLIND\d{2}$")
    total: Optional[float]
    status: Literal["completed", "partial", "failed"]


class EvalOutput(BaseModel):
    """eval_<팀번호>.json 최상위 구조. results 외 다른 키는 검증 단계에서 즉시 실패한다."""

    model_config = ConfigDict(extra="forbid")

    results: list[ResultItem]

## 5. 골드셋 · 답변 파일 로더

공통 필드(`id`/`question`/`gold_articles`/`key_facts`)만 신뢰하고
`ptype`/`difficulty` 등 추가 필드에는 의존하지 않습니다.

In [ ]:
import glob
import json
import re
from dataclasses import dataclass
from typing import Any

_BLIND_ID_PATTERN = re.compile(r"^BLIND\d{2}$")  # 검증용 (완전 일치)
_BLIND_ID_SEARCH_PATTERN = re.compile(r"BLIND\d{2}")  # 파일명에서 부분 탐색용


@dataclass(frozen=True)
class GoldQuestion:
    id: str
    question: str
    gold_articles: list[dict[str, Any]]
    key_facts: list[str]


@dataclass(frozen=True)
class AnswerItem:
    qid: str
    retrieved: list[list[Any]]
    answer: str


@dataclass(frozen=True)
class AnswerFile:
    blind_id: str
    path: str
    answers: dict[str, AnswerItem]  # qid -> AnswerItem


class LoaderError(Exception):
    """입력 파일 검증 실패. 이 예외를 잡은 상위 로직은 해당 팀을 failed 처리한다."""


def load_gold_questions(path: str) -> list[GoldQuestion]:
    """공통 필드만 추출한다. _meta, ptype, difficulty, tag 등은 무시한다."""
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)

    questions = raw.get("questions")
    if not isinstance(questions, list) or not questions:
        raise LoaderError(f"골드셋에 questions 배열이 없거나 비어 있습니다: {path}")

    result: list[GoldQuestion] = []
    seen_ids: set[str] = set()
    for q in questions:
        qid = q.get("id")
        if not qid:
            raise LoaderError(f"gold 문항에 id가 없습니다: {q}")
        if qid in seen_ids:
            raise LoaderError(f"gold 문항 id가 중복되었습니다: {qid}")
        seen_ids.add(qid)

        gold_articles = q.get("gold_articles")
        if not isinstance(gold_articles, list) or not gold_articles:
            raise LoaderError(f"{qid}: gold_articles가 없거나 비어 있습니다")

        key_facts = q.get("key_facts")
        if not isinstance(key_facts, list) or not key_facts:
            raise LoaderError(f"{qid}: key_facts가 없거나 비어 있습니다")

        result.append(
            GoldQuestion(
                id=qid,
                question=q.get("question", ""),
                gold_articles=gold_articles,
                key_facts=key_facts,
            )
        )
    return result


def load_answer_file(path: str, blind_id_override: Optional[str] = None) -> AnswerFile:
    """
    답변 파일 하나를 읽는다.
    blind_id_override가 주어지면 그 값을 쓰고(개발 단계 테스트용),
    아니면 파일 내부 필드(blind_id/team) -> 파일명 순서로 찾는다.
    """
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)

    blind_id = blind_id_override or raw.get("blind_id") or raw.get("team")
    if not blind_id:
        m = _BLIND_ID_SEARCH_PATTERN.search(path.upper())
        if m:
            blind_id = m.group(0)
    if not blind_id:
        raise LoaderError(f"blind_id를 파일에서도, 파일명에서도, 인자로도 찾을 수 없습니다: {path}")

    answers_raw = raw.get("answers")
    if not isinstance(answers_raw, list):
        raise LoaderError(f"{path}: answers 배열이 없습니다")

    answers: dict[str, AnswerItem] = {}
    for a in answers_raw:
        qid = a.get("qid")
        if not qid:
            continue  # qid 없는 항목은 스킵, 문항 단위 결측으로 처리됨(aggregate 단계)
        if qid in answers:
            raise LoaderError(f"{path}: qid가 중복되었습니다: {qid}")
        answers[qid] = AnswerItem(
            qid=qid,
            retrieved=a.get("retrieved", []) or [],
            answer=a.get("answer", "") or "",
        )

    return AnswerFile(blind_id=str(blind_id), path=path, answers=answers)


def discover_answer_files(pattern: str) -> list[str]:
    """glob 패턴으로 파일 경로를 찾는다. 문항 수·팀 수를 코드에 박지 않기 위한 장치."""
    paths = sorted(glob.glob(pattern))
    if not paths:
        raise LoaderError(f"패턴에 해당하는 파일이 없습니다: {pattern}")
    return paths


def is_valid_blind_id(blind_id: str) -> bool:
    """BLIND01~BLIND99 형식인지만 확인한다. 개발 단계 team 필드("3" 등)는 False."""
    return bool(_BLIND_ID_PATTERN.match(blind_id))


def validate_blind_ids(answer_files: list[AnswerFile]) -> None:
    """
    중복·형식 오류를 검증한다. 누락(5개 미만)은 여기서 에러로 보지 않는다.
    누락 여부는 순위 산정 단계(운영진)의 관심사이기 때문이다.
    """
    seen: set[str] = set()
    for af in answer_files:
        if not _BLIND_ID_PATTERN.match(af.blind_id):
            raise LoaderError(f"blind_id 형식이 올바르지 않습니다: {af.blind_id} ({af.path})")
        if af.blind_id in seen:
            raise LoaderError(f"blind_id가 중복되었습니다: {af.blind_id}")
        seen.add(af.blind_id)

## 6. 채점 — 근거 조항 (MRR, 20점 만점 중 0~1 정규화)

[1단계] 표준 MRR, 문서명·조번호 완전 일치만 인정합니다.
2단계에서 `_normalize_key`만 교체하면(문서명 정규화 + 조번호 파서 + any-match)
`score_evidence` 본문은 그대로 씁니다.

In [ ]:
def score_evidence(gold_articles: list[dict[str, Any]], retrieved: list[list[Any]]) -> float:
    """
    gold_articles: [{"doc": "카카오계정 약관", "article": 10, "citation": "..."}, ...]
    retrieved:     [["카카오계정 약관", 10], ...] (최대 4개)

    반환: MRR (0.0~1.0). gold가 여러 개(OR 관계, 예: P02)여도 하나만 찾으면
          만점이 되도록 gold 전체를 대상으로 최고 순위를 찾는다.
    """
    if not retrieved:
        return 0.0

    gold_keys = {_normalize_key(g.get("doc"), g.get("article")) for g in gold_articles}

    for rank, item in enumerate(retrieved[:4], start=1):
        if len(item) < 2:
            continue
        doc, article = item[0], item[1]
        if _normalize_key(doc, article) in gold_keys:
            return 1.0 / rank

    return 0.0


def _normalize_key(doc: Any, article: Any) -> tuple[str, str]:
    """1단계: 앞뒤 공백만 제거한 문자열 비교. 2단계에서 정규화 사전 + 조번호 파서로 교체."""
    doc_norm = str(doc).strip() if doc is not None else ""
    article_norm = str(article).strip() if article is not None else ""
    return (doc_norm, article_norm)

## 7. 채점 — 답변 내용 (F1, 30점 만점 중 0~1 정규화)

[1단계] 어절(공백) 단위 토큰 F1. 반환 타입을 처음부터 `dict`로 잡아 둔 것이 확장 훅입니다.
2단계에서 `detail`에 극성 반전(`polarity_flip`) 같은 플래그를 채우기만 하면
호출부(`aggregate_question`)는 바뀌지 않습니다.

In [ ]:
from collections import Counter


def score_content(key_facts: list[str], answer: str) -> dict:
    """
    key_facts: 정답 핵심 문장 리스트
    answer:    참가팀 답변 원문
    반환: {"score": f1(0~1), "detail": {}}
    """
    gold_text = " ".join(key_facts)
    gold_tokens = Counter(_tokenize(gold_text))
    pred_tokens = Counter(_tokenize(answer))

    overlap = sum((gold_tokens & pred_tokens).values())
    if overlap == 0:
        return {"score": 0.0, "detail": {}}

    precision = overlap / sum(pred_tokens.values())
    recall = overlap / sum(gold_tokens.values())
    f1 = 2 * precision * recall / (precision + recall)

    return {"score": f1, "detail": {}}


def _tokenize(text: str) -> list[str]:
    """1단계: 공백 기준 어절 분리. 2단계에서 형태소 분석기(Kiwi 등)로 교체 예정."""
    return [t for t in text.strip().split() if t]

## 8. 채점 — LLM 평가 (정확성·근거성·완결성·명료성, 50점 만점)

모델은 Gemini 3.5 Flash 하나만 씁니다. SDK는 신규 `google-genai`를 사용합니다
(구 `google-generativeai`는 지원이 종료되었습니다). `classify_type`/`cache_key`는
이미 호출되고 정의되어 있지만 1단계에서는 로직이 비어 있는 스텁입니다 — 2/3단계에서
이 두 함수 내부만 채우면 나머지 코드는 손댈 필요가 없습니다.

In [ ]:
import hashlib

# 정확한 모델 ID는 Google AI Studio / API 문서에서 최신값을 확인해서 채워 넣을 것.
# 환경변수로 오버라이드 가능하게 해서, 모델 ID가 바뀌어도 코드 수정 없이 대응한다.
_DEFAULT_MODEL_NAME = "gemini-3.5-flash"

# 루브릭/프롬프트 문구를 바꿀 때마다 이 값을 올린다. cache_key에 포함되므로
# 버전이 다르면 캐시가 자동으로 무효화된다(3단계에서 실제로 캐시에 연결).
PROMPT_VERSION = "v1"


def score_llm(
    question: str,
    gold_articles: list[dict[str, Any]],
    key_facts: list[str],
    answer: str,
    blind_id: str = "",
    qid: str = "",
) -> dict:
    """LLM 채점 진입점. 실패 시 예외를 던진다 — 상위(run)에서 잡아 partial 처리."""
    qtype = classify_type(question, key_facts)
    prompt = build_prompt(question, gold_articles, key_facts, answer, qtype)

    _ = cache_key(blind_id, qid, PROMPT_VERSION, prompt)  # 1단계는 호출만, 저장/조회는 안 함

    raw = _call_gemini(prompt)
    return _parse_response(raw)


def classify_type(question: str, key_facts: list[str]) -> str:
    """
    문항 유형 자동 분류 (골드셋 ptype 필드에 의존하지 않는다 — 비공개셋엔 없을 수 있음).
    1단계 스텁: 항상 "basic"을 반환한다.
    2단계에서 채울 규칙:
      - 질문이 "~나요/~인가요"로 끝나고 key_facts[0]이 부정어로 시작 -> "polarity"
      - 숫자·법조문·기간 패턴 포함 -> "numeric"
      - "가지"가 있고 key_facts 4개 이상 -> "enumerate"
      - "순서로"/"먼저" -> "procedure"
      - 그 외 -> "basic"
    """
    return "basic"


def build_prompt(
    question: str,
    gold_articles: list[dict[str, Any]],
    key_facts: list[str],
    answer: str,
    qtype: str = "basic",
) -> str:
    """1단계: 유형 구분 없이 단일 템플릿. 2단계에서 qtype별 "특히 확인할 것" 절만 분기."""
    citation_lines = "\n".join(
        f"- {g.get('citation') or g.get('doc')}" for g in gold_articles
    )
    facts_lines = "\n".join(f"{i + 1}. {f}" for i, f in enumerate(key_facts))

    return f"""당신은 카카오 약관 기반 RAG 답변을 채점하는 평가자입니다.
아래 질문에 대한 참가팀의 답변을 정답과 비교하여 4개 항목을 평가하세요.

[질문]
{question}

[근거 조항]
{citation_lines}

[정답 핵심 내용]
{facts_lines}

[참가팀 답변]
{answer}

[평가 항목 — 각 0~4점]
정확성(accuracy): 답변의 사실 주장이 정답과 일치하는가.
근거성(groundedness): 답변 내용이 근거 조항 범위 내에 있는가(지어낸 내용이 없는가).
완결성(completeness): 정답 핵심 내용을 빠짐없이 다루었는가.
명료성(clarity): 구조와 표현이 읽기 쉬운가.

다음 JSON만 출력하세요. 다른 말은 쓰지 마세요.
{{"accuracy": <0-4>, "groundedness": <0-4>, "completeness": <0-4>, "clarity": <0-4>, "reason": "<40자 이내>"}}"""


def cache_key(blind_id: str, qid: str, prompt_version: str, prompt: str) -> str:
    """3단계에서 실제 캐시 조회/저장에 쓸 키. prompt_version이 바뀌면 캐시가 자동 무효화된다."""
    raw = f"{blind_id}|{qid}|{prompt_version}|{prompt}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


def _call_gemini(prompt: str) -> str:
    """Gemini API 호출. 키는 환경변수(GOOGLE_API_KEY)에서만 읽는다.

    구 SDK(google-generativeai)는 지원이 종료되어 신규 SDK(google-genai)를 사용한다.
    """
    from google import genai
    from google.genai import types

    api_key = os.environ.get("GOOGLE_API_KEY")
    if not api_key:
        raise RuntimeError(
            "GOOGLE_API_KEY 환경변수가 설정되어 있지 않습니다. "
            "2번 셀(API 키 주입)을 먼저 확인하세요."
        )

    client = genai.Client(api_key=api_key)
    model_name = os.environ.get("GEMINI_MODEL_NAME", _DEFAULT_MODEL_NAME)

    response = client.models.generate_content(
        model=model_name,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.0,
            response_mime_type="application/json",
        ),
    )
    return response.text


def _parse_response(raw: str) -> dict:
    """JSON 파싱 + 필드/범위 검증. 형식이 어긋나면 예외를 던져 partial 처리로 넘긴다."""
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        cleaned = cleaned.split("\n", 1)[-1] if "\n" in cleaned else cleaned

    data = json.loads(cleaned)

    required = ("accuracy", "groundedness", "completeness", "clarity")
    for key in required:
        if key not in data:
            raise ValueError(f"LLM 응답에 {key} 필드가 없습니다: {data}")
        if not (0 <= data[key] <= 4):
            raise ValueError(f"LLM 응답 {key} 값이 범위를 벗어났습니다: {data[key]}")

    data.setdefault("reason", "")
    return data

## 9. 집계 — 문항 단위 + 팀 단위

배점: 근거(MRR) 20 + 내용(F1) 30 + LLM 50.

결측/실패 처리 규칙:
- 팀이 특정 qid에 답을 아예 안 낸 경우 -> **0점 처리**하고 평균 분모에 포함 (제외하면
  답을 적게 낼수록 유리해지는 역인센티브가 생김)
- LLM 호출이 실패한 문항 -> 근거+내용(50점 만점) 기준으로 재정규화, 해당 팀은 `partial`
- 입력 파일 자체가 깨진 경우 -> `run()`에서 `LoaderError`를 잡아 `failed` 처리

In [ ]:
from dataclasses import dataclass as _dataclass  # 이름 충돌 방지용 별칭


EVIDENCE_WEIGHT = 20.0
CONTENT_WEIGHT = 30.0
LLM_WEIGHT = 50.0

# LLM 4항목 내부 가중치 (정확성 40%·근거성 25%·완결성 20%·명료성 15%)
_LLM_SUBWEIGHTS = {
    "accuracy": 0.40,
    "groundedness": 0.25,
    "completeness": 0.20,
    "clarity": 0.15,
}


@_dataclass
class QuestionScore:
    qid: str
    score_0_100: float
    llm_failed: bool


def llm_subscore(llm_result: dict) -> float:
    """LLM 4항목(각 0~4)을 0~1로 정규화한 가중합."""
    weighted = sum(_LLM_SUBWEIGHTS[key] * llm_result[key] for key in _LLM_SUBWEIGHTS)
    return weighted / 4.0


def aggregate_question(
    qid: str,
    evidence_score: float,
    content_result: dict,
    llm_result: Optional[dict],
) -> QuestionScore:
    """한 문항의 0~100점을 계산한다."""
    content_score = content_result["score"]

    if llm_result is not None:
        earned = (
            evidence_score * EVIDENCE_WEIGHT
            + content_score * CONTENT_WEIGHT
            + llm_subscore(llm_result) * LLM_WEIGHT
        )
        return QuestionScore(qid=qid, score_0_100=earned, llm_failed=False)

    # LLM 실패: 근거+내용(50점 만점)만으로 재정규화
    earned_without_llm = evidence_score * EVIDENCE_WEIGHT + content_score * CONTENT_WEIGHT
    max_without_llm = EVIDENCE_WEIGHT + CONTENT_WEIGHT
    renormalized = (earned_without_llm / max_without_llm) * 100.0
    return QuestionScore(qid=qid, score_0_100=renormalized, llm_failed=True)


def aggregate_team(question_scores: list[QuestionScore]) -> tuple[Optional[float], str]:
    """팀 전체 total/status를 결정한다."""
    if not question_scores:
        return None, "failed"

    total = sum(qs.score_0_100 for qs in question_scores) / len(question_scores)
    status = "partial" if any(qs.llm_failed for qs in question_scores) else "completed"
    return total, status

## 10. 저장 + 재검증

저장 직후 파일을 다시 읽어 `EvalOutput`으로 재검증합니다. 금지 필드 혼입은 여기서 즉시 잡힙니다.

In [ ]:
def save_eval_output(path: str, results: list[ResultItem]) -> None:
    output = EvalOutput(results=results)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(output.model_dump(), f, ensure_ascii=False, indent=2)

    _revalidate(path)


def _revalidate(path: str) -> None:
    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    if set(data.keys()) != {"results"}:
        raise AssertionError(f"최상위 키가 results만이어야 합니다. 실제: {sorted(data.keys())}")

    EvalOutput(**data)  # 필드 누락/타입 오류/extra 필드는 여기서 즉시 예외

## 11. 실행

골드셋과 답변 파일들을 찾아 팀별로 채점하고 저장합니다.
LLM 호출 실패는 문항 단위로 격리해 `partial`로 흡수하며, 이 셀 자체는 예외를 던지지 않습니다.

In [ ]:
def evaluate_team(gold_questions: list[GoldQuestion], answer_file: AnswerFile) -> ResultItem:
    question_scores: list[QuestionScore] = []

    for gq in gold_questions:
        answer_item = answer_file.answers.get(gq.id)

        # 결측 문항: 0점 처리하되 평균 분모에는 포함한다 (역인센티브 방지).
        retrieved = answer_item.retrieved if answer_item else []
        answer_text = answer_item.answer if answer_item else ""

        evidence_score = score_evidence(gq.gold_articles, retrieved)
        content_result = score_content(gq.key_facts, answer_text)

        llm_result = None
        try:
            llm_result = score_llm(
                question=gq.question,
                gold_articles=gq.gold_articles,
                key_facts=gq.key_facts,
                answer=answer_text,
                blind_id=answer_file.blind_id,
                qid=gq.id,
            )
        except Exception as exc:  # noqa: BLE001 — 문항 단위 실패 격리가 목적
            print(f"[경고] {answer_file.blind_id} {gq.id} LLM 채점 실패: {exc}")

        question_scores.append(
            aggregate_question(gq.id, evidence_score, content_result, llm_result)
        )

    total, status = aggregate_team(question_scores)
    return ResultItem(blind_id=answer_file.blind_id, total=total, status=status)


def run() -> None:
    gold_paths = discover_answer_files(GOLD_GLOB)
    gold_questions = load_gold_questions(gold_paths[0])
    print(f"골드셋 로드: {gold_paths[0]} ({len(gold_questions)}문항)")

    answer_paths = discover_answer_files(ANSWER_GLOB)
    print(f"답변 파일 {len(answer_paths)}개 발견")

    results: list[ResultItem] = []
    for path in answer_paths:
        try:
            answer_file = load_answer_file(path, blind_id_override=DEV_BLIND_ID_OVERRIDE)
        except LoaderError as exc:
            print(f"[실패] {path} 로딩 실패: {exc}")
            continue

        if not is_valid_blind_id(answer_file.blind_id):
            print(
                f"[스킵] {path}: blind_id \'{answer_file.blind_id}\'가 BLIND01~05 형식이 "
                "아닙니다. 실전 파일이 아니라면 3번 셀의 DEV_BLIND_ID_OVERRIDE를 설정하세요."
            )
            continue

        try:
            result = evaluate_team(gold_questions, answer_file)
        except Exception as exc:  # noqa: BLE001 — 팀 단위 완전 실패
            print(f"[실패] {answer_file.blind_id} 채점 중단: {exc}")
            result = ResultItem(blind_id=answer_file.blind_id, total=None, status="failed")

        results.append(result)
        print(f"  {result.blind_id}: total={result.total} status={result.status}")

    save_eval_output(OUTPUT_PATH, results)
    print(f"저장 완료: {OUTPUT_PATH}")


run()